# Missingness-aware VaDeSC vs. imputation on SUPPORT (3 splits x 5 seeds)

Notebook 06 compared the missingness-aware VaDeSC with mean imputation on one data split (3 seeds), and notebook 10 compared median/mode with VAE imputation on 3 splits x 5 seeds. Here all of them are run together, on the same splits and seeds as notebook 10, so every VaDeSC variant has a paired run with the same split and seed.

SUPPORT without the post-baseline variables. VaDeSC variants (all with the same small network, K = 4, 16 latent dimensions, Weibull k = 2, 300 epochs):

| | missing numeric values | missing categories | input | reconstruction loss |
|---|---|---|---|---|
| **median/mode imputation** | training median | most frequent category | values | all values |
| **VAE imputation** | VAE (notebook 10) | VAE | values | all values |
| **mean imputation** | training mean (0 after scaling) | own "missing" category | values | all values |
| **+ mask input** | training mean | own "missing" category | values and 0/1 mask | all values and the mask |
| **MissVaDeSC** | training mean | own "missing" category | values and mask | observed values only |
| **MissVaDeSC + mask model** | training mean | own "missing" category | values and mask | observed values, plus $p(m \mid z)$ |

Cox models with the same inputs are the simple references. Full run: about 15 min on a CPU (three VAE fits, then the VaDeSC runs as 10 parallel jobs).

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed
from scipy import stats

from imputation.vae_impute import load_support_from_frame, vae_impute_support
from src.baselines import run_cox
from src.data import load_support
from src.missing import MissVaDeSC, fill_and_mask
from src.vadesc import evaluate_vadesc, train_vadesc

QUICK = False
SPLITS = [42, 1] if QUICK else [42, 1, 2]   # same as notebook 10
SEEDS = [0, 1] if QUICK else [0, 1, 2, 3, 4]
EPOCHS = 60 if QUICK else 300
N_JOBS = 10

## 1. Data for each split

In [2]:
data = {}
for sp in SPLITS:
    keep = load_support(drop_post_baseline=True, seed=sp, keep_missing=True)
    data[(sp, "median/mode imputation")] = load_support(drop_post_baseline=True, seed=sp)
    data[(sp, "VAE imputation")] = load_support_from_frame(
        vae_impute_support(drop_post_baseline=True, split_seed=sp, verbose=True), drop_post_baseline=True, seed=sp)
    data[(sp, "mean imputation")] = fill_and_mask(keep, False)
    data[(sp, "mask")] = fill_and_mask(keep, True)
print({k: v.X_tr.shape for k, v in data.items() if k[0] == 42})
print(f"missing numeric entries in the training set (split 42): {np.isnan(load_support(drop_post_baseline=True, keep_missing=True).X_tr).mean():.1%}")

VAE (Encoder Indicators, split 42): 134 epochs, best valid ELBO -27.024


VAE (Encoder Indicators, split 1): 185 epochs, best valid ELBO -26.002


VAE (Encoder Indicators, split 2): 162 epochs, best valid ELBO -26.384


{(42, 'median/mode imputation'): (6373, 51), (42, 'VAE imputation'): (6373, 51), (42, 'mean imputation'): (6373, 53), (42, 'mask'): (6373, 106)}
missing numeric entries in the training set (split 42): 7.2%


## 2. Cox references

In [3]:
cox = []
for sp in SPLITS:
    for name, key in [("Cox, median/mode imputation", "median/mode imputation"), ("Cox, VAE imputation", "VAE imputation"),
                      ("Cox, mean imputation", "mean imputation"), ("Cox + mask", "mask")]:
        m = run_cox(data[(sp, key)])
        cox.append({"split": sp, "method": name, "C-index": m["C-index"], "IBS": m["IBS"]})
cox = pd.DataFrame(cox)
cox.groupby("method", sort=False)[["C-index", "IBS"]].agg(["mean", "std"]).round(3)

C-index           IBS       
                               mean    std   mean    std
method                                                  
Cox, median/mode imputation   0.723  0.009  0.179  0.004
Cox, VAE imputation           0.723  0.009  0.178  0.005
Cox, mean imputation          0.723  0.009  0.179  0.005
Cox + mask                    0.724  0.010  0.179  0.005

## 3. VaDeSC variants (3 splits x 5 seeds each)

In [4]:
VARIANTS = {
    "VaDeSC, median/mode imputation": ("median/mode imputation", {}),
    "VaDeSC, VAE imputation": ("VAE imputation", {}),
    "VaDeSC, mean imputation": ("mean imputation", {}),
    "VaDeSC + mask input": ("mask", {}),
    "MissVaDeSC": ("mask", {"model_cls": MissVaDeSC, "model_kwargs": {"model_mask": False, "net": "small"}}),
    "MissVaDeSC + mask model": ("mask", {"model_cls": MissVaDeSC, "model_kwargs": {"model_mask": True, "net": "small"}}),
}


def run_one(name, sp, s):
    torch.set_num_threads(2)
    key, kw = VARIANTS[name]
    kw = {"model_kwargs": {"net": "small"}, **kw}  # the same small network for every variant
    d = data[(sp, key)]
    m, _, _ = evaluate_vadesc(train_vadesc(d, K=4, d_lat=16, weibull_k=2.0, epochs=EPOCHS, seed=s, **kw), d)
    return {"split": sp, "seed": s, "method": name, "C-index": m["C-index"], "IBS": m["IBS"],
            "log-rank chi2": m["log-rank chi2"]}


jobs = [(name, sp, s) for sp in SPLITS for s in SEEDS for name in VARIANTS]
res = pd.DataFrame(Parallel(n_jobs=N_JOBS, verbose=5)(delayed(run_one)(*j) for j in jobs))
pd.concat([res, cox.assign(seed=np.nan)]).to_csv("../results/missingness_support_splits.csv", index=False)
res.groupby("method", sort=False)[["C-index", "IBS", "log-rank chi2"]].agg(["mean", "std"]).round(3)

[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.


[Parallel(n_jobs=10)]: Done  52 tasks      | elapsed:  6.6min


[Parallel(n_jobs=10)]: Done  90 out of  90 | elapsed: 10.5min finished


C-index           IBS        log-rank chi2  \
                                  mean    std   mean    std          mean   
method                                                                      
VaDeSC, median/mode imputation   0.656  0.016  0.206  0.006       239.537   
VaDeSC, VAE imputation           0.667  0.018  0.202  0.006       294.291   
VaDeSC, mean imputation          0.677  0.018  0.196  0.006       318.669   
VaDeSC + mask input              0.659  0.015  0.200  0.006       267.436   
MissVaDeSC                       0.662  0.014  0.198  0.005       296.170   
MissVaDeSC + mask model          0.652  0.009  0.202  0.005       275.327   

                                        
                                   std  
method                                  
VaDeSC, median/mode imputation  46.592  
VaDeSC, VAE imputation          70.834  
VaDeSC, mean imputation         78.422  
VaDeSC + mask input             70.351  
MissVaDeSC                      81.354  
MissVaDeSC + mask model         86.205

## 4. Paired differences

Each variant minus VaDeSC with median/mode imputation (the setup used in the rest of the report), and each mask variant minus VaDeSC with mean imputation (the same filled-in values, so only the handling of missingness differs). Same split and seed; 95% CI over the 15 pairs. The runs within a split share the test set, so the per-split means are also shown.

In [5]:
wide = res.pivot_table(index=["split", "seed"], columns="method", values="C-index")


def paired(a, b):
    diff = wide[a] - wide[b]
    n = len(diff)
    ci = stats.t.ppf(0.975, n - 1) * diff.std(ddof=1) / np.sqrt(n)
    per_split = diff.groupby(level="split").mean()
    return {"comparison": f"{a}  minus  {b}", "mean diff": diff.mean(), "95% CI": f"[{diff.mean() - ci:+.3f}, {diff.mean() + ci:+.3f}]",
            "p-value": stats.ttest_1samp(diff, 0).pvalue, "better in": f"{(diff > 0).sum()}/{n} runs",
            "per-split diffs": per_split.round(3).tolist()}


ref = "VaDeSC, median/mode imputation"
comps = [(m, ref) for m in VARIANTS if m != ref] + \
        [(m, "VaDeSC, mean imputation") for m in ["VaDeSC + mask input", "MissVaDeSC", "MissVaDeSC + mask model"]] + \
        [("MissVaDeSC", "VaDeSC, VAE imputation")]
pd.set_option("display.max_colwidth", 80)
pd.DataFrame([paired(a, b) for a, b in comps]).round(3)

,comparison,mean diff,95% CI,p-value,better in,per-split diffs
0,"VaDeSC, VAE imputation minus VaDeSC, median/mode imputation",0.012,"[+0.000, +0.023]",0.050,12/15 runs,"[0.009, 0.018, 0.008]"
1,"VaDeSC, mean imputation minus VaDeSC, median/mode imputation",0.021,"[+0.011, +0.031]",0.000,12/15 runs,"[0.022, 0.023, 0.018]"
2,"VaDeSC + mask input minus VaDeSC, median/mode imputation",0.004,"[-0.007, +0.015]",0.482,8/15 runs,"[0.01, 0.006, -0.005]"
3,"MissVaDeSC minus VaDeSC, median/mode imputation",0.006,"[-0.002, +0.014]",0.131,10/15 runs,"[0.008, 0.006, 0.004]"
4,"MissVaDeSC + mask model minus VaDeSC, median/mode imputation",-0.003,"[-0.012, +0.006]",0.452,7/15 runs,"[0.001, -0.004, -0.007]"
5,"VaDeSC + mask input minus VaDeSC, mean imputation",-0.017,"[-0.031, -0.004]",0.016,4/15 runs,"[-0.012, -0.017, -0.023]"
6,"MissVaDeSC minus VaDeSC, mean imputation",-0.015,"[-0.026, -0.004]",0.011,4/15 runs,"[-0.014, -0.017, -0.014]"
7,"MissVaDeSC + mask model minus VaDeSC, mean imputation",-0.024,"[-0.034, -0.015]",0.000,1/15 runs,"[-0.021, -0.027, -0.025]"
8,"MissVaDeSC minus VaDeSC, VAE imputation",-0.006,"[-0.017, +0.006]",0.307,7/15 runs,"[-0.001, -0.012, -0.004]"


In [6]:
# per split (VaDeSC averaged over seeds)
res.groupby(["method", "split"], sort=False)["C-index"].mean().unstack("split").round(3)

split,42,1,2
method,,,
"VaDeSC, median/mode imputation",0.670,0.645,0.653
"VaDeSC, VAE imputation",0.678,0.654,0.670
"VaDeSC, mean imputation",0.688,0.667,0.676
VaDeSC + mask input,0.665,0.655,0.659
MissVaDeSC,0.673,0.653,0.659
MissVaDeSC + mask model,0.663,0.646,0.649


## 5. Correction for multiple comparisons

Section 4 makes several paired comparisons, so some of them can look significant by chance. Here all of them, plus mean imputation vs. VAE imputation, are recomputed from the saved results with a Bonferroni correction (p-value times the number of comparisons). A difference that has the same sign in all three splits but does not pass the correction is only "consistently lower/higher", not clearly different.

In [ ]:
res = pd.read_csv("../results/missingness_support_splits.csv")
res = res[res.method.str.contains("VaDeSC")]
wide = res.pivot_table(index=["split", "seed"], columns="method", values="C-index")


def paired(a, b):
    diff = wide[a] - wide[b]
    n = len(diff)
    ci = stats.t.ppf(0.975, n - 1) * diff.std(ddof=1) / np.sqrt(n)
    return {"comparison": f"{a}  minus  {b}", "mean diff": diff.mean(),
            "95% CI": f"[{diff.mean() - ci:+.3f}, {diff.mean() + ci:+.3f}]",
            "p-value": stats.ttest_1samp(diff, 0).pvalue, "better in": f"{(diff > 0).sum()}/{n} runs",
            "per-split diffs": diff.groupby(level="split").mean().round(3).tolist()}


ref = "VaDeSC, median/mode imputation"
comps = [(m, ref) for m in wide.columns if m != ref] +         [(m, "VaDeSC, mean imputation") for m in ["VaDeSC + mask input", "MissVaDeSC", "MissVaDeSC + mask model"]] +         [("MissVaDeSC", "VaDeSC, VAE imputation"), ("VaDeSC, mean imputation", "VaDeSC, VAE imputation")]
out = pd.DataFrame([paired(a, b) for a, b in comps])
out["p (Bonferroni)"] = (out["p-value"] * len(comps)).clip(upper=1)
print(f"{len(comps)} paired comparisons, Bonferroni threshold: p < {0.05 / len(comps):.4f} (uncorrected)")
out.round(4)

10 paired comparisons, Bonferroni threshold: p < 0.0050 (uncorrected)


,comparison,mean diff,95% CI,p-value,better in,per-split diffs,p (Bonferroni)
0,"MissVaDeSC minus VaDeSC, median/mode imputation",0.0060,"[-0.002, +0.014]",0.1307,10/15 runs,"[0.008, 0.006, 0.004]",1.0000
1,"MissVaDeSC + mask model minus VaDeSC, median...",-0.0033,"[-0.012, +0.006]",0.4522,7/15 runs,"[0.001, -0.004, -0.007]",1.0000
2,"VaDeSC + mask input minus VaDeSC, median/mod...",0.0038,"[-0.007, +0.015]",0.4815,8/15 runs,"[0.01, 0.006, -0.005]",1.0000
3,"VaDeSC, VAE imputation minus VaDeSC, median/...",0.0117,"[+0.000, +0.023]",0.0499,12/15 runs,"[0.009, 0.018, 0.008]",0.4992
4,"VaDeSC, mean imputation minus VaDeSC, median...",0.0212,"[+0.011, +0.031]",0.0005,12/15 runs,"[0.022, 0.023, 0.018]",0.0049
5,"VaDeSC + mask input minus VaDeSC, mean imput...",-0.0174,"[-0.031, -0.004]",0.0158,4/15 runs,"[-0.012, -0.017, -0.023]",0.1584
6,"MissVaDeSC minus VaDeSC, mean imputation",-0.0151,"[-0.026, -0.004]",0.0109,4/15 runs,"[-0.014, -0.017, -0.014]",0.1090
7,"MissVaDeSC + mask model minus VaDeSC, mean i...",-0.0245,"[-0.034, -0.015]",0.0001,1/15 runs,"[-0.021, -0.027, -0.025]",0.0008
8,"MissVaDeSC minus VaDeSC, VAE imputation",-0.0056,"[-0.017, +0.006]",0.3065,7/15 runs,"[-0.001, -0.012, -0.004]",1.0000
9,"VaDeSC, mean imputation minus VaDeSC, VAE im...",0.0095,"[+0.000, +0.019]",0.0470,12/15 runs,"[0.013, 0.006, 0.01]",0.4700


## Notes

SUPPORT without the post-baseline variables, 3 splits x 5 seeds (15 paired runs per VaDeSC variant). 10 paired comparisons in total; with a Bonferroni correction a difference needs p < 0.005 (section 5).

- **Cox does not care how the missing values are handled**: 0.723-0.724 with median/mode, VAE or mean imputation and with the mask as extra features.
- **Mean imputation is clearly better than median/mode imputation for VaDeSC**: 0.677 ± 0.018 vs 0.656 ± 0.016 (+0.021, 95% CI [0.011, 0.031], better in all three splits; Bonferroni p = 0.005, at the threshold). Mean imputation here also puts missing categorical values in their own category.
- **Mean imputation vs VAE imputation**: nominally higher (0.677 vs 0.667, paired +0.010, 95% CI [0.000, 0.019], uncorrected p = 0.047), but not significant after the correction.
- **VAE imputation vs median/mode** (same comparison as notebook 10, rerun here): +0.012, 95% CI [0.000, 0.023], better in all three splits, but uncorrected p = 0.050, so not significant after the correction. Slightly smaller than in notebook 10 (+0.016); VaDeSC runs are not bit-identical between the two notebooks (parallel training here).
- **Handling missingness inside the model does not help on SUPPORT.** Compared with VaDeSC with mean imputation (the same filled-in values), MissVaDeSC with the mask model is clearly worse (-0.024, Bonferroni p < 0.001). Giving the mask as input (-0.017) and MissVaDeSC (-0.015) are consistently lower (in all three splits), but do not pass the correction (uncorrected p = 0.016 and 0.011). Compared with median/mode imputation all three are about the same.
- Why mean imputation beats median/mode is not tested here: the two differ both in the fill value for numeric variables (mean vs median) and in the categorical variables (own "missing" category vs most frequent category).
- Results: `results/missingness_support_splits.csv`.